# King Domino Projekt - Overblik

### Arkitekturoversigt
Projektets formål er at digitalisere og point-score fysiske King Domino-spilleplader. Projektet består af fem hovedfaser:
1. **Gitteropdeling (Grid Splitting):** 
    - Opdeling af hele pladebilleder i et 5x5 logisk gitter af terræn-tiles.
2. **Feature-ekstraktion:** 
    - Behandling af tærren-tiles for at udtrække features til en 20-værdi vektor (HSV-histogram logik)
    - Gemme 20-værdi vektorerne i en csv-fil til modeltræning/analyse.
3. **Tile-multiklassificering:** 
    - Identificering af tile-terræntypen (Water, Field, Grass, Forest, Mine, Swamp, Home, Empty). 
    - Altså en multi-class terrænklassificering udført med kNN / SVM med robust stringente prædefinerede 5-fold grouped splits.
4. **Krone-detektion:** 
    - Tælle antallet af score-multiplikatorer (kroner) til stede på hver terræn-tile ved brug af template matching og hyperparameter-tunet "Non-Maximum Suppression (NMS)".
5. **Spillogik og Scoreberegning:** 
    - Evaluering af 2D-arrays (5x5).
    - Breadth-First Search (BFS) logik til at udregne spillets endelige score. (Regel: points for region = region_size * crowns).
6. **Ground Truths(GT):**
    - GT_BoardPointScore (Done!)
    - GT_BoardTileMatrix (Done!)
    - GT_CrownsPerBoard (Done!)
    - GT_CrownsPerTile (Done!)



### Nuværende status

| Modul | Status | Beskrivelse |
| :--- | :---: | :--- |
| `board_split.py` / `save_tiles.py`| 🟢 | Deler spilleplader op i et 5x5 tile-gitter rent. |
| `feature_extraction.py` | 🟢 | Udtrækker 20-værdi vektor HSV-histogrammer fra tiles og gemmer i (`features.csv`). |
| `classify_tile.py` / `svm_clssifier.py` | 🟢 | Modeller er implementeret med stringent cross-validation (`PredefinedSplit`) for at eliminere datalækage. |
| `detect_crown.py` / `template_matching.py` | 🟢 | Fuld end-to-end krone identifikation med GUI og NMS thresholding bygget. |
| `scoring.py` | 🟢 | `explore_region` algoritme klar og understøtter korrekte Kingdomino BFS-søgningsregler. |
| End-to-end integration (`evaluate_pipeline.py`) | 🟢 | Kæden integreres nu fejlfrit til "in-memory" inferens uden mellemliggende `.csv`-udveksling og evaluerer hold-out sættet med MAE-metrikker. |


### Manglende dele & Anbefalede næste skridt
For at afslutte maskinlærings-rejsen fuldstændig, er følgende opgaver de resterende:
1. **Model Persistence / Model Eksport:** Ekstrahere den trænede SVM-model vha. `pickle` eller `joblib`, så applikationen kan læse den direkte i RAM og undgå omtræning hver gang `evaluate_pipeline.py` køres.
2. **Evaluering / Illustrationer:** Udregne og optegne Confusion Matrix (CM), Error Heatmaps eller Precision-Recall Curve til at understøtte rapport-valideringen.
3. **Rapportering:** Skrive King Domino opgaven i dybden, som opridset i "To do.txt".

## 1. Dataforbehandling og split-strategi
For at undgå datalækage laves split konsekvent på spilleplade-niveau og ikke på tile-niveau. Parrede billeder (med og uden 3D-objekter) placeres altid i samme split.

* **Samlet antal spilleplader:** 36 unikke spilleplader.
* **Ugyldige data(støj):** Billede 71, 73 og 74 er udeladt, fordi de er dubletter af billede 68, 69 og 70.
* **Dataekstraktion:** ca. 1775 tiles (71 gyldige billeder × 25 felter).

### Endelig split:
* **Hold-out testsæt:** 6 spilleplade-grupper (12 billeder i alt) reserveres til den endelige evaluering for at teste præstation af ML-model.

Test-sæt:
Spilleplader
  - (1,5), (19,23), (25,29), (35,39), (49,53), (67,70)

* **Træning og validering (grouped CV):** De resterende 30 spilleplade-grupper bruges til 5-fold grouped cross-validation.
  * Hver fold indeholder 6 spilleplade-grupper.
  * I hver CV-runde bruges 4 folds til træning og 1 fold til validering.

Fold 1:
Spilleplader 
  - Mixed_data: (4,8), (20,24), (34,38), (42,46), (48,52), (65,72)
  - Uden3dObjekt(clean_data): 3, 20, 34, 42, 48, 65
  - Med3dObjekt(dirty_data): 8, 24, 38, 46, 52, 72

Fold 2:
Spilleplader
(2,6), (18,22), (28,32), (36,40), (51,55), (58,62)

Fold 3:
Spilleplader
(10,14), (11,15), (26,30), (41,44), (57,61), (64,68)

Fold 4:
Spilleplader
(3,7), (17,21), (27,31), (43,47), (50,54), (59,63)

Fold 5:
Spilleplader
(9,13), (12,16), (33,37), (45), (56,60), (66,69)
